In [1]:
from datasets import load_dataset, concatenate_datasets, load_from_disk, DownloadMode
from torch.utils.data import DataLoader
from torch.nn import Linear, Embedding, Parameter
import torch.optim as optim
import torch
import numpy as np
import random
import os
from tqdm.auto import tqdm
import spacy
import re
from collections import Counter
from torchtext.vocab import vocab, FastText
from collections import OrderedDict
from typing import List, Dict

/home/anwesh/scratch/miniconda3/envs/pt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed = 5758):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

cache_location = '/home/anwesh/scratch/hf_cache/'
dataset = load_dataset('roneneldan/TinyStories', cache_dir=cache_location)

# dataset['train'][0]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [3]:
os.environ['SPACY_DATA'] = '/home/anwesh/scratch/spacy_data/'
# os.environ['SPACY_DATA'] = '/home/anwesh/scratch/spacy_data/'
full_dataset = concatenate_datasets([dataset['train'], dataset['validation']])

print("Full dataset size:", len(full_dataset))

Full dataset size: 2141709


In [4]:
dataset['train'][0]

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}

In [5]:
_spacy_model = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'textcat'])

def spacy_tokenize(text):
    """Tokenize a single string using spaCy."""
    return [token.text.lower() for token in _spacy_model(text) if not token.is_space]

def batch_spacy_tokenize(batch):
    """Tokenize a batch of samples using spaCy's nlp.pipe."""
    texts = batch['text']
    return {'tokens': [[token.text.lower() for token in doc if not token.is_space] for doc in _spacy_model.pipe(texts, batch_size=512*4, n_process=1)]}

full_dataset = concatenate_datasets([dataset['train'], dataset['validation']])


In [13]:
len(full_dataset)

2141709

In [6]:
# tokenized_full = full_dataset.map(batch_spacy_tokenize, batched=True, batch_size=512*4, remove_columns=['text'], num_proc=8)
# print("Tokenization complete.")

tokenized_full = load_from_disk("/home/anwesh/scratch/ELL8299 Project/tokenized_tiny_stories_data")
print("Tokenized dataset loaded from disk.")

Tokenized dataset loaded from disk.


In [ ]:
train_size = len(dataset['train']) 
valid_size = len(dataset['validation'])  

In [9]:
len(dataset['train']), len(dataset['validation'])

(2119719, 21990)

In [11]:
tokenized_train = tokenized_full.select(range(0, train_size))
tokenized_valid = tokenized_full.select(range(train_size, train_size + valid_size))

In [12]:
len(tokenized_train), len(tokenized_valid)

(2119719, 21990)

In [9]:
# all_tokens = [token for tokens in tokenized_full['tokens'] for token in tokens]
# special_tokens = ['<unk>', '<pad>', '<sos>', '<eos>']
# vocab_counter = Counter(all_tokens)
# tiny_stories_vocab = vocab(vocab_counter, specials=special_tokens, min_freq=5, specials_first=True)
# tiny_stories_vocab.set_default_index(tiny_stories_vocab['<unk>'])

tiny_stories_vocab = torch.load("/home/anwesh/scratch/ELL8299 Project/tiny_stories_vocab.pt")
tiny_stories_vocab.set_default_index(tiny_stories_vocab['<unk>'])

In [10]:
fasttext_vectors = FastText(language='en', cache='/home/anwesh/scratch/ELL8299 Project/vector_cache')

In [11]:
VOCAB_SIZE = len(tiny_stories_vocab)        
EMBEDDING_DIM = 300

In [12]:
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
words_found = 0

# Initialize with small random values
embedding_matrix = np.random.normal(scale=0.6, size=(VOCAB_SIZE, EMBEDDING_DIM))

for idx, token in enumerate(tiny_stories_vocab.get_itos()):
    try:
        # Get the vector for the token from FastText
        vector = fasttext_vectors.get_vecs_by_tokens(token)
        embedding_matrix[idx] = vector
        words_found += 1
    except KeyError:
        # For tokens not in FastText (like <unk>, <pad>, etc.), the random initialization is kept
        pass

print(f"Found {words_found} / {VOCAB_SIZE} words in FastText vocabulary.")

# Explicitly set padding token embedding to zeros
pad_idx = tiny_stories_vocab['<pad>']
embedding_matrix[pad_idx] = np.zeros(EMBEDDING_DIM)

Found 32787 / 32787 words in FastText vocabulary.


In [13]:
embedding_layer = Embedding.from_pretrained(
    torch.tensor(embedding_matrix, dtype=torch.float32),  # Explicitly set to float32
    freeze=False     # Set to False if you want to fine-tune the embeddings
)

In [14]:
class PositionalEncoding(torch.nn.Module):

    """
    Implements the sinusoidal positional encoding as described in the "Attention is All You Need" paper.
    This module adds positional information to the input embeddings to help the model understand the
    order of tokens.
    
    Args:
        d_model (int): The dimension of the embeddings.
        max_length (int): The maximum length of the input sequences.
    """

    def __init__(self, d_model: int = 256, max_length : int = 64):
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model

        # position term
        pos = torch.arange(0, max_length).unsqueeze(1)

        ##10000^(2i/d_model)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))

        # init the positional encoding matrix
        pe = torch.zeros(max_length, d_model)
        
        ##even terms in embedding dim
        pe[:, 0::2] = torch.sin(pos * div_term)

        ##odd terms in embedding dim
        pe[:, 1::2] = torch.cos(pos * div_term)

        pe = pe.unsqueeze(0)  # shape: (1, max_length, d_model)

        ## Non trainable, so register as buffer
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)

        Returns:
            Tensor of shape (batch_size, seq_len, d_model) with positional encodings added
        """
        x = x * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return x

In [15]:
class LayerNorm(torch.nn.Module):

    """    
    Implementation of Layer Normalization as described in the "Layer Normalization" paper.
    
    Args:
        features (int): The number of features in the input tensor.
        eps (float): A small value to avoid division by zero during normalization.
    """

    def __init__(self, features: int, eps: float = 1e-6):
        super(LayerNorm, self).__init__()
        self.gamma = Parameter(torch.ones(features))
        self.beta = Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, features)

        Returns:
            Tensor of the same shape as input with layer normalization applied
        """
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta



In [16]:
class MultiHeadAttention(torch.nn.Module):

    """ 
    Implementation of Multi-Head Attention mechanism as described in the "Attention is All You Need" paper.

    Args:
        d_model (int): The dimension of the input embeddings.
        num_heads (int): The number of attention heads.
        dropout (float): Dropout rate to apply after attention.
    """

    def __init__(self, d_model: int = 256, num_heads: int = 8, dropout: float = 0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.linear_q = Linear(d_model, d_model)
        self.linear_k = Linear(d_model, d_model)
        self.linear_v = Linear(d_model, d_model)
        self.linear_out = Linear(d_model, d_model)

        self.dropout = torch.nn.Dropout(dropout)
        self.softmax = torch.nn.Softmax(dim=-1)

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            query: Tensor of shape (batch_size, seq_len, d_model)
            key: Tensor of shape (batch_size, seq_len, d_model)
            value: Tensor of shape (batch_size, seq_len, d_model)
            mask: Optional tensor for masking (batch_size, seq_len, seq_len)

        Returns:
            Tensor of shape (batch_size, seq_len, d_model) after applying multi-head attention
        """
        batch_size = query.size(0)

        ##The reshaping and transposing below enables parallel computation of attention across multiple heads

        q = self.linear_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.linear_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.linear_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn_weights = self.softmax(scores)
        attn_weights = self.dropout(attn_weights)

        attn_output = torch.matmul(attn_weights, v)

        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # Final linear layer
        output = self.linear_out(attn_output)
        return output

In [17]:
class ResidualConnection(torch.nn.Module):

    """
    Implements a residual connection followed by layer normalization.
    
    Args:
        size (int): The number of features in the input tensor.
        dropout (float): Dropout rate to apply after the residual connection.
    """

    def __init__(self, size: int, dropout: float = 0.2):
        super(ResidualConnection, self).__init__()
        self.layer_norm = LayerNorm(size)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, sublayer: torch.nn.Module) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, features)
            sublayer: A sublayer module to apply to the input

        Returns:
            Tensor of the same shape as input after applying residual connection and layer normalization
        """
        return x + self.dropout(sublayer(self.layer_norm(x)))

In [18]:
class FFN(torch.nn.Module):

    """
    Implements the Position-wise Feed-Forward Network as described in the "Attention is All You Need" paper.
    
    Args:
        d_model (int): The dimension of the input embeddings.
        d_ff (int): The dimension of the feed-forward layer.
        dropout (float): Dropout rate to apply after the feed-forward layer.
    """

    def __init__(self, d_model: int = 256, d_ff: int = 1024, dropout: float = 0.1):
        super(FFN, self).__init__()
        self.linear1 = Linear(d_model, d_ff)
        self.linear2 = Linear(d_ff, d_model)
        self.dropout = torch.nn.Dropout(dropout)
        self.relu = torch.nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)

        Returns:
            Tensor of shape (batch_size, seq_len, d_model) after applying feed-forward network
        """
        return self.linear2(self.dropout(self.relu(self.linear1(x))))

In [19]:
class DecoderBlock(torch.nn.Module):

    """    
    Implementation of a Transformer Decoder Block as described in the "Attention is All You Need" paper.
    
    Args:
        d_model (int): The dimension of the input embeddings.
        num_heads (int): The number of attention heads.
        d_ff (int): The dimension of the feed-forward network.
        dropout (float): Dropout rate to apply in various parts of the block.
    """

    def __init__(self, d_model: int = 256, num_heads: int = 8, d_ff: int = 512, dropout: float = 0.1):
        super(DecoderBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)

        self.feed_forward = torch.nn.Sequential(
            Linear(d_model, d_ff),
            torch.nn.ReLU(),
            Linear(d_ff, d_model)
        )
        self.residual2 = ResidualConnection(d_model, dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len, d_model)
            mask: Optional tensor for masking (batch_size, seq_len, seq_len)

        Returns:
            Tensor of shape (batch_size, seq_len, d_model) after applying the decoder block
        """
        x = self.residual1(x, lambda x: self.self_attn(x, x, x, mask))
        x = self.residual2(x, self.feed_forward)
        return x

In [20]:
class DecoderTransformer(torch.nn.Module):

    """
    Implementation of a Transformer Decoder as described in the "Attention is All You Need" paper.
    
    Args:
        vocab_size (int): The size of the vocabulary.
        d_model (int): The dimension of the input embeddings.
        num_heads (int): The number of attention heads.
        d_ff (int): The dimension of the feed-forward network.
        num_layers (int): The number of decoder layers.
        max_length (int): The maximum length of the input sequences.
        dropout (float): Dropout rate to apply in various parts of the model.
        pretrained_embeddings (Embedding): Optional pre-trained embedding layer.
    """

    def __init__(self, vocab_size: int, d_model: int = 256, num_heads: int = 8, d_ff: int = 512, num_layers: int = 3, max_length: int = 64, dropout: float = 0.1, pretrained_embeddings: Embedding = None):
        super(DecoderTransformer, self).__init__()

        # Use pretrained embeddings if provided, otherwise create new ones
        if pretrained_embeddings is not None:
            self.embedding = pretrained_embeddings
            # If pretrained embeddings have different dimension, add a projection layer
            if pretrained_embeddings.embedding_dim != d_model:
                self.embedding_projection = Linear(pretrained_embeddings.embedding_dim, d_model)
            else:
                self.embedding_projection = None
        else:
            self.embedding = Embedding(vocab_size, d_model)
            self.embedding_projection = None
            
        self.positional_encoding = PositionalEncoding(d_model, max_length)

        self.layers = torch.nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.layer_norm = LayerNorm(d_model)
        self.output_linear = Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (batch_size, seq_len)
            mask: Optional tensor for masking (batch_size, seq_len, seq_len)

        Returns:
            Tensor of shape (batch_size, seq_len, vocab_size) after applying the decoder transformer
        """

        batch_size, seq_len = x.size()

        # Create causal mask if not provided
        if mask is None:
            # Create mask with shape (1, 1, seq_len, seq_len) for broadcasting
            mask = torch.triu(torch.ones((1, 1, seq_len, seq_len), device=x.device), diagonal=1).bool()

        x = self.embedding(x)
        
        # Project embeddings if dimensions don't match
        if self.embedding_projection is not None:
            x = self.embedding_projection(x)
            
        x = self.positional_encoding(x)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.layer_norm(x)
        output = self.output_linear(x)
        return output

In [21]:
def create_decoder_transformer(seq_len: int, device: str, pretrained_embeddings: Embedding = None) -> DecoderTransformer:

    model = DecoderTransformer(
        vocab_size=len(tiny_stories_vocab),
        d_model=256,
        num_heads=8,
        d_ff=512,
        num_layers=3,
        max_length=seq_len,
        dropout=0.1,
        pretrained_embeddings=pretrained_embeddings
    ).to(device)
    
    return model

seq_len = 64
decoder_model = create_decoder_transformer(
    seq_len=seq_len, 
    device='cuda' if torch.cuda.is_available() else 'cpu',
    pretrained_embeddings=embedding_layer
)

In [22]:
def collate_fn(batch: List[Dict], vocab: vocab, seq_len: int) -> Dict[str, torch.Tensor]:
    """Custom collate function to prepare batches of data."""
    input_ids = []
    target_ids = []

    for sample in batch:
        tokens = sample['tokens']
        token_ids = [vocab['<sos>']] + [vocab[token] for token in tokens] + [vocab['<eos>']]
        
        # Truncate or pad sequences to seq_len
        if len(token_ids) > seq_len:
            token_ids = token_ids[:seq_len]
        else:
            token_ids += [vocab['<pad>']] * (seq_len - len(token_ids))
        
        input_ids.append(token_ids[:-1])  # Input sequence
        target_ids.append(token_ids[1:])  # Target sequence (shifted by 1)

    return {
        'input_ids': torch.tensor(input_ids, dtype=torch.long),
        'target_ids': torch.tensor(target_ids, dtype=torch.long)
    }

def create_dataloaders(train_dataset, valid_dataset, vocab, seq_len, batch_size):
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda batch: collate_fn(batch, vocab, seq_len)
    )

    valid_dataloader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda batch: collate_fn(batch, vocab, seq_len)
    )

    return train_dataloader, valid_dataloader
    

In [28]:
train_loader, valid_loader = create_dataloaders(tokenized_train, tokenized_valid, tiny_stories_vocab, seq_len, batch_size=512)

In [ ]:
tokenized_train

In [29]:
def train_decoder_transformer(decoder_transformer=decoder_model, max_seq_len=seq_len, train_loader=train_loader,
                              validation_loader=valid_loader, lr=3e-4, weight_decay=0, num_epochs=10):
    """
    Train the decoder transformer model.
    
    Args:
        decoder_transformer: The model to train
        max_seq_len: Maximum sequence length
        train_loader: Training DataLoader
        validation_loader: Validation DataLoader
        lr: Learning rate
        weight_decay: Weight decay for optimizer
        num_epochs: Number of training epochs
    """
    optimizer = optim.Adam(decoder_transformer.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=tiny_stories_vocab['<pad>'])
    
    device = next(decoder_transformer.parameters()).device

    for epoch in range(num_epochs):
        # Training phase
        decoder_transformer.train()
        total_train_loss = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['target_ids'].to(device)

            optimizer.zero_grad()
            output = decoder_transformer(input_ids)

            output = output.view(-1, output.size(-1))
            target_ids = target_ids.view(-1)

            loss = criterion(output, target_ids)
            loss.backward()
            
            # Clip gradients to prevent them from exploding
            torch.nn.utils.clip_grad_norm_(decoder_transformer.parameters(), max_norm=1.0)
            
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        
        # Validation phase
        decoder_transformer.eval()
        total_val_loss = 0
        
        with torch.no_grad():
            for batch in tqdm(validation_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]"):
                input_ids = batch['input_ids'].to(device)
                target_ids = batch['target_ids'].to(device)

                output = decoder_transformer(input_ids)

                output = output.view(-1, output.size(-1))
                target_ids = target_ids.view(-1)

                loss = criterion(output, target_ids)
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(validation_loader)
        
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        ##At the end of the epoch print an input, trrget and output sample in text format

        sample_batch = next(iter(validation_loader))
        input_ids = sample_batch['input_ids'].to(device)
        target_ids = sample_batch['target_ids'].to(device)
        output = decoder_transformer(input_ids)
        predicted_ids = torch.argmax(output, dim=-1)
        for i in range(1):
            input_text = ' '.join([tiny_stories_vocab.get_itos()[idx] for idx in input_ids[i].cpu().numpy() if idx != tiny_stories_vocab['<pad>']])
            target_text = ' '.join([tiny_stories_vocab.get_itos()[idx] for idx in target_ids[i].cpu().numpy() if idx != tiny_stories_vocab['<pad>']])
            predicted_text = ' '.join([tiny_stories_vocab.get_itos()[idx] for idx in predicted_ids[i].cpu().numpy() if idx != tiny_stories_vocab['<pad>']])
            print(f"\nSample {i+1}:\nInput: {input_text}\nTarget: {target_text}\nPredicted: {predicted_text}\n")
    


In [30]:
def eval_decoder_transformer(decoder_transformer=decoder_model, test_loader=valid_loader):
    """
    Evaluate the decoder transformer model.
    
    Args:
        decoder_transformer: The model to evaluate
        test_loader: Test/validation DataLoader
        
    Returns:
        Average loss on the test set
    """
    decoder_transformer.eval()
    criterion = torch.nn.CrossEntropyLoss(ignore_index=tiny_stories_vocab['<pad>'])
    device = next(decoder_transformer.parameters()).device
    
    total_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            target_ids = batch['target_ids'].to(device)

            output = decoder_transformer(input_ids)

            output = output.view(-1, output.size(-1))
            target_ids = target_ids.view(-1)

            loss = criterion(output, target_ids)
            total_loss += loss.item()
    
    avg_loss = total_loss / len(test_loader)
    print(f"Test Loss: {avg_loss:.4f}")
    
    return avg_loss

In [ ]:
train_decoder_transformer(decoder_transformer=decoder_model, lr=3e-4, weight_decay=0, num_epochs=10)

Epoch 1/10 [Valid]: 100%|██████████| 43/43 [00:06<00:00,  7.00it/s]



Epoch 1/10 - Train Loss: 2.0259, Val Loss: 1.7614

Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water
Target: spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water .
Predicted: once was spot is a big thing . wanted , " wow , look ! look car is so pretty ! pretty ! " spot said and said , " i you , spot ! i am my ! day . " spot a with the car , spot went spot ran happy . spot looked a big , with water water .


Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and re

Epoch 2/10 [Valid]: 100%|██████████| 43/43 [00:06<00:00,  6.99it/s]



Epoch 2/10 - Train Loss: 1.7614, Val Loss: 1.6628

Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water
Target: spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water .
Predicted: once was he was a big thing and wanted , " wow , that ! that car is so pretty ! shiny . " spot was and said , " yes you , i ! i love my ! day . " spot a , his car , spot went his went very . he went a big bottle with water water and


Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and rep

Epoch 3/10 [Valid]: 100%|██████████| 43/43 [00:06<00:00,  6.76it/s]



...

Epoch 8/10 - Train Loss: 1.5863, Val Loss: 1.5366

Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water
Target: spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water .
Predicted: once was he was a big thing and he , " i , that ! that car is so pretty ! shiny ! " his was and said , " yes you , spot . " 'm it . day . " spot a for his car , spot went spot went so . they went a nice pond with water water .


Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied ,

Epoch 9/10 [Valid]: 100%|██████████| 43/43 [00:06<00:00,  6.78it/s]


Epoch 9/10 - Train Loss: 1.5771, Val Loss: 1.5286

Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water
Target: spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water .
Predicted: once was he was a big thing and wanted , " i , that ! that car is so pretty ! shiny ! " spot smiled and said , " yes you " spot ! i 'm it . day . " spot a for the car , spot went spot went so . they went a nice bottle with water water .



Epoch 10/10 [Valid]: 100%|██████████| 43/43 [00:06<00:00,  6.99it/s]


Epoch 10/10 - Train Loss: 1.5694, Val Loss: 1.5221

Sample 1:
Input: <sos> spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water
Target: spot . spot saw the shiny car and said , " wow , kitty , your car is so bright and clean ! " kitty smiled and replied , " thank you , spot . i polish it every day . " after playing with the car , kitty and spot felt thirsty . they found a small pond with clear water .
Predicted: once was he was a big thing and he , " i , that ! that car is so shiny ! shiny ! " spot was and said , " yes you , spot ! i 'm it with day . " spot a for his car , spot went spot went so . they went a big pond and water water .

